# Tutorial 3h: Designing a Double Gauss Camera Lens from Flat Plates


### June 2025


The **Double Gauss** is the masterpiece of classical photographic optics.  Since Alvan Clark
patented the first Double Gauss objective in 1888 and Carl Zeiss introduced the Planar in
1896, virtually every fast normal camera lens has been a variation on this design.  The
near-symmetric arrangement of two negative cemented doublets around a central stop is the
key: odd aberrations (coma, distortion, lateral colour) are cancelled by symmetry, leaving
the designer free to correct the even residuals (spherical, astigmatism, field curvature,
axial colour) with the remaining degrees of freedom.

In this tutorial we start from literally nothing — **six flat glass plates** — and reach a
well-corrected F/4 Double Gauss through a three-phase workflow that mirrors how
professional lens designers approach the problem from scratch:

1. **Seidel correction** — bend the plates into a recognisable Double Gauss shape with the
   right focal length and balanced primary (third-order) aberrations.
2. **Polychromatic RMS spot optimisation** — switch to real-ray polychromatic merit to push
   beyond the Seidel approximation.
3. **Air-gap refinement** — free the inter-element spacings to squeeze out the final
   performance.

After optimisation we analyse the design with a full set of professional diagnostics: spot
diagrams, ray fans, RMS wavefront error vs field, MTF, field curvature, and distortion.

**Design target**

| Parameter | Value |
|-----------|-------|
| Effective focal length | 100 mm |
| F-number | F/4 (EPD = 25 mm) |
| Half field of view | 0°, 10°, 15° |
| Wavelengths | 486.1 nm (F), 587.6 nm (d, primary), 656.3 nm (C) |
| Elements | 6 (two positive singlets + two cemented negative doublets) |

This tutorial assumes familiarity with Tutorial 3g (Cooke triplet case study) and Tutorial
3c (DLS mechanics).  If you are new to Optiland optimization start there.


In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt

from optiland import optic, analysis
from optiland.optimization import minimize, OptimizationProblem
from optiland.optimization.observers.history import HistoryObserver
from optiland.optimization.stopping.criteria import MaxIter, CostTolerance

import optiland.backend as be

## 1. Build the Starting System

The Double Gauss consists of:

- **E1** — front positive singlet (N-SK16 crown)
- **E2 + E3** — front cemented negative doublet (N-SK16 crown / N-SF5 flint)
- Aperture stop (in air, between the two doublets)
- **E4 + E5** — rear cemented negative doublet (N-SF5 flint / N-SK16 crown)
- **E6** — rear positive singlet (N-SK16 crown)

The system is approximately symmetric front-to-rear about the stop.  That symmetry is
what cancels odd aberrations and makes the Double Gauss such a powerful starting topology.

We begin with every surface radius set to 10^10 mm — optically indistinguishable from a
flat plate.  Thicknesses and glass types are fixed at physically reasonable values;
the optimizer will establish curvature from zero.

Surface layout:

```
idx  material   thickness   role
  0  —           inf        object at infinity
  1  N-SK16      8.0 mm     E1 front surface
  2  air         0.5 mm     E1 rear / air gap to front doublet
  3  N-SK16     12.0 mm     E2 front surface (crown of cemented doublet)
  4  N-SF5       4.0 mm     cemented interface (flint of front doublet)
  5  air        16.0 mm     rear of front doublet / air gap to stop
  6  air        14.0 mm     aperture stop surface
  7  N-SF5       4.0 mm     E4 front surface (flint of rear doublet)
  8  N-SK16     12.0 mm     cemented interface (crown of rear doublet)
  9  air         0.5 mm     E5 rear / air gap to E6
 10  N-SK16      8.0 mm     E6 front surface
 11  air        62.0 mm     back focal distance (becomes a variable in Phase 2)
 12  —           —          image plane
```


In [ ]:
lens = optic.Optic()

r_flat = 1e10  # effectively flat: 1/r ~ 0

# Object at infinity
lens.surfaces.add(index=0, thickness=np.inf)

# E1 — front positive crown singlet
lens.surfaces.add(index=1,  thickness=8.0,  radius=r_flat, material='N-SK16')

# Air gap between E1 and front cemented doublet
lens.surfaces.add(index=2,  thickness=0.5,  radius=r_flat)

# E2+E3 — front cemented doublet (crown then flint)
lens.surfaces.add(index=3,  thickness=12.0, radius=r_flat, material='N-SK16')  # E2 crown
lens.surfaces.add(index=4,  thickness=4.0,  radius=r_flat, material='N-SF5')   # E3 flint (cemented)

# Air gap from front doublet to aperture stop
lens.surfaces.add(index=5,  thickness=16.0, radius=r_flat)

# Aperture stop — sits in air, carries no optical power
lens.surfaces.add(index=6,  thickness=14.0, radius=np.inf, is_stop=True)

# E4+E5 — rear cemented doublet (flint then crown) — mirror of front doublet
lens.surfaces.add(index=7,  thickness=4.0,  radius=r_flat, material='N-SF5')   # E4 flint
lens.surfaces.add(index=8,  thickness=12.0, radius=r_flat, material='N-SK16')  # E5 crown (cemented)

# Air gap from rear doublet to E6
lens.surfaces.add(index=9,  thickness=0.5,  radius=r_flat)

# E6 — rear positive crown singlet (symmetric counterpart to E1)
lens.surfaces.add(index=10, thickness=8.0,  radius=r_flat, material='N-SK16')

# Back focal distance — will become a variable in Phase 2
lens.surfaces.add(index=11, thickness=62.0, radius=r_flat)

# Image plane
lens.surfaces.add(index=12)

# System specification
lens.set_aperture(aperture_type='EPD', value=25.0)       # F/4 at EFL = 100 mm
lens.fields.set_type('angle')
lens.fields.add(y=0.0)                                   # on-axis
lens.fields.add(y=10.0)                                  # 67% field
lens.fields.add(y=15.0)                                  # full field
lens.wavelengths.add(value=0.4861)                       # F line (blue)
lens.wavelengths.add(value=0.5876, is_primary=True)      # d line (yellow-green)
lens.wavelengths.add(value=0.6563)                       # C line (red)

print(f'Starting EFL: {lens.paraxial.f2():.3e} mm  (effectively infinite — all plates are flat)')


In [ ]:
# Save a deep copy of the starting state for before/after comparisons
starting = copy.deepcopy(lens)

print('=== Starting design: six flat glass plates ===')
lens.draw()


## 2. Phase 1 — Seidel Correction (Curvature and Focal Length)

The first task is to establish optical power: bend the flat plates until the system has an
effective focal length of 100 mm, while simultaneously driving the five primary (third-order)
Seidel aberrations to zero.

**Variables:** the reciprocal radius (1/*r*) of the **eight air-glass interfaces** —
surfaces 1, 2, 3, 5, 7, 9, 10, and 11.  The two cemented interfaces (surfaces 4 and 8)
are deliberately kept flat at this stage.  Their index step is small (N-SK16 / N-SF5,
Δ*n* ≈ 0.08) so their refractive contribution is minor, but if the Seidel optimizer
drives them to extreme curvatures they can cause total-internal-reflection failures
in later real-ray phases.  Releasing them in Phase 2, once the design has physical
geometry, avoids this pitfall.

**Operands:** effective focal length f₂ = 100 mm, plus Seidel coefficients SI–SV = 0.
Eight variables for six operands leaves the problem well-conditioned with two degrees of
freedom to spare.

The image plane position is *not* a variable here because Seidel aberrations are
independent of the image distance.  After optimisation we use `image_solve()` to
automatically move the image surface to the paraxial focus.


In [ ]:
problem1 = OptimizationProblem()

# Paraxial focal length target
problem1.add_operand('f2', target=100.0, weight=1.0, input_data={'optic': lens})

# Five primary Seidel aberrations to zero:
#   1 = spherical  2 = coma  3 = astigmatism  4 = Petzval field curvature  5 = distortion
for i in range(1, 6):
    problem1.add_operand('seidel', target=0.0, weight=1.0,
                         input_data={'optic': lens, 'seidel_number': i})

# Reciprocal radii of the eight air-glass interfaces only.
# Cemented interfaces (surfaces 4, 8) remain flat in this phase to prevent the Seidel
# optimizer from assigning them extreme curvatures that would cause TIR in Phase 2.
for s in [1, 2, 3, 5, 7, 9, 10, 11]:
    problem1.add_variable(lens, 'reciprocal_radius', surface_number=s,
                          min_val=-0.01, max_val=0.01)

print(f'Variables: {len(problem1.variables)}')
print(f'Operands:  {len(problem1.operands)}')
problem1.info()


In [ ]:
result1 = minimize(problem1, 'dls', stop=MaxIter(100))
print(result1)

In [ ]:
# Move the image surface to the paraxial focus of the current design
lens.updater.image_solve()

print(f'Phase 1 EFL:            {lens.paraxial.f2():.2f} mm')
print(f'Phase 1 image distance: {lens.surfaces[11].thickness:.2f} mm')
print(f'Phase 1 Seidels:        {lens.aberrations.seidels()}')

print()
print('=== Design after Phase 1 (Seidel) ===')
lens.draw()

In [ ]:
# Save a snapshot of the Phase 1 design for the final comparison table
lens_p1 = copy.deepcopy(lens)

print('=== Spot diagram after Phase 1 (Seidel design, d-line) ===')
spot_p1 = analysis.SpotDiagram(lens_p1)
spot_p1.view()


The Seidel step has bent the flat plates into a recognisable Double Gauss shape and
established a 100 mm focal length.  The spot diagram, however, still shows significant
residual aberration — Seidel theory is a third-order approximation, and the real ray
performance at F/2.8 is dominated by higher-order terms that the first-order Seidel
analysis cannot control.

Phase 2 switches the merit function to actual traced-ray performance.


## 3. Phase 2 — Polychromatic RMS Spot Optimisation

We now replace the Seidel operands with polychromatic RMS spot-size operands.  Passing
`wavelength='all'` to `rms_spot_size` traces rays at all three design wavelengths
simultaneously, using the primary (d-line) centroid as the reference point — the standard
polychromatic spot-size definition used in commercial design codes.

Two new variables are added in this phase:

- **Cemented interface curvatures** (surfaces 4 and 8): now that the design has a
  physically reasonable shape, we release the cemented interfaces with conservative bounds
  (|1/*r*| ≤ 0.04, minimum radius ≈ 25 mm) to let the optimizer tune the doublet power
  split and chromatic correction.
- **Image distance** (surface 11 thickness): the Seidel phase placed the image at the
  paraxial focus; Phase 2 will find the best *real-ray* focus.

We attach a `HistoryObserver` to plot the convergence curve after the run.


In [ ]:
problem2 = OptimizationProblem()

# Keep the focal length on target
problem2.add_operand('f2', target=100.0, weight=1.0, input_data={'optic': lens})

# Polychromatic RMS spot size at each field point
# wavelength='all' traces F, d, and C simultaneously; reference centroid = d-line
for Hx, Hy in lens.fields.get_field_coords():
    problem2.add_operand('rms_spot_size', target=0.0, weight=1.0, input_data={
        'optic': lens,
        'surface_number': -1,
        'Hx': Hx,
        'Hy': Hy,
        'num_rays': 5,
        'wavelength': 'all',
        'distribution': 'hexapolar',
    })

# All eight air-glass interfaces from Phase 1
for s in [1, 2, 3, 5, 7, 9, 10, 11]:
    problem2.add_variable(lens, 'reciprocal_radius', surface_number=s)

# Cemented interfaces — now released with conservative bounds (min radius ~25 mm)
problem2.add_variable(lens, 'reciprocal_radius', surface_number=4, min_val=-0.04, max_val=0.04)
problem2.add_variable(lens, 'reciprocal_radius', surface_number=8, min_val=-0.04, max_val=0.04)

# Image distance is now free
problem2.add_variable(lens, 'thickness', surface_number=11, min_val=40.0, max_val=120.0)

print(f'Variables: {len(problem2.variables)}')
print(f'Operands:  {len(problem2.operands)}')
problem2.info()


In [ ]:
history_obs = HistoryObserver()
stop2 = MaxIter(200) | CostTolerance(1e-8)

result2 = minimize(problem2, 'dls', stop=stop2, observers=[history_obs])
print(result2)


In [ ]:
_ = lens.draw()

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy([r['value'] for r in result2.history], color='steelblue', linewidth=1.5)
plt.xlabel('Iteration')
plt.ylabel('Merit function (log scale)')
plt.title('Phase 2 — polychromatic RMS spot convergence')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Save Phase 2 snapshot
lens_p2 = copy.deepcopy(lens)

print('=== Spot diagram after Phase 2 (polychromatic RMS spot) ===')
spot_p2 = analysis.SpotDiagram(lens_p2)
spot_p2.view()

Phase 2 has substantially tightened the spots.  The on-axis field is already approaching
near-diffraction-limited performance; the residual error at full field is dominated by
higher-order astigmatism and coma.

The element spacings (air gaps) have been fixed throughout Phases 1 and 2.  Phase 3
releases them, giving the optimizer an additional four degrees of freedom to redistribute
aberration between the groups.


## 4. Phase 3 — Air-Gap Refinement

The inter-element spacings are a powerful handle in Double Gauss design.  The gap between
the front doublet and the stop controls the balance between the front- and rear-group
aberration contributions; the symmetric rear gap mirrors it.  The small gap between E1 and
the front doublet (and its mirror after E5) influences the marginal-ray height on each
element and therefore the relative aberration weighting.

We now free four air thicknesses in addition to the ten curvatures and the image distance,
for a total of **15 variables** against four operands (EFL + three polychromatic RMS spots).
The system is heavily over-determined in the variable space, which gives the DLS algorithm
excellent conditioning.


In [ ]:
problem3 = OptimizationProblem()

problem3.add_operand('f2', target=100.0, weight=1.0, input_data={'optic': lens})

for Hx, Hy in lens.fields.get_field_coords():
    problem3.add_operand('rms_spot_size', target=0.0, weight=1.0, input_data={
        'optic': lens,
        'surface_number': -1,
        'Hx': Hx,
        'Hy': Hy,
        'num_rays': 5,
        'wavelength': 'all',
        'distribution': 'hexapolar',
    })

# All ten surface curvatures
for s in [1, 2, 3, 5, 7, 9, 10, 11]:
    problem3.add_variable(lens, 'reciprocal_radius', surface_number=s)
problem3.add_variable(lens, 'reciprocal_radius', surface_number=4, min_val=-0.04, max_val=0.04)
problem3.add_variable(lens, 'reciprocal_radius', surface_number=8, min_val=-0.04, max_val=0.04)

# Air gaps — bounded to prevent physically impossible configurations
problem3.add_variable(lens, 'thickness', surface_number=2,  min_val=0.5,  max_val=5.0)   # E1 to front doublet
problem3.add_variable(lens, 'thickness', surface_number=5,  min_val=5.0,  max_val=25.0)  # front doublet to stop
problem3.add_variable(lens, 'thickness', surface_number=6,  min_val=5.0,  max_val=25.0)  # stop to rear doublet
problem3.add_variable(lens, 'thickness', surface_number=9,  min_val=0.5,  max_val=5.0)   # rear doublet to E6

# Image distance
problem3.add_variable(lens, 'thickness', surface_number=11, min_val=40.0, max_val=120.0)

print(f'Variables: {len(problem3.variables)}')
print(f'Operands:  {len(problem3.operands)}')
problem3.info()

In [ ]:
result3 = minimize(problem3, 'dls', stop=MaxIter(300) | CostTolerance(1e-9))
print(result3)

In [ ]:
# Phase 3 snapshot for the comparison table
lens_p3 = copy.deepcopy(lens)
spot_p3 = analysis.SpotDiagram(lens_p3)

# Collect RMS spot radii from each phase (d-line, wavelength index 1)
rms1 = spot_p1.rms_spot_radius()
rms2 = spot_p2.rms_spot_radius()
rms3 = spot_p3.rms_spot_radius()

fields = lens.fields.get_field_coords()
wav_primary = 1  # d-line is the second wavelength added (index 1)

print(f'{"".ljust(65, "=")}')
print('  RMS spot radius — d-line (mm)              ')
print(f'{"".ljust(65, "-")}')
print(f'  {"Field":>8}   {"Phase 1":>12}   {"Phase 2":>12}   {"Phase 3":>12}')
print(f'  {"-"*8}   {"-"*12}   {"-"*12}   {"-"*12}')
for i, (_, Hy) in enumerate(fields):
    r1 = float(be.to_numpy(rms1[i][wav_primary]))
    r2 = float(be.to_numpy(rms2[i][wav_primary]))
    r3 = float(be.to_numpy(rms3[i][wav_primary]))
    print(f'  Hy = {Hy:>4.1f}°  {r1:>12.5f}   {r2:>12.5f}   {r3:>12.5f}')
print(f'{"".ljust(65, "=")}')
print()
print(f'Final EFL:            {lens.paraxial.f2():.2f} mm')
print(f'Final image distance: {lens.surfaces[11].thickness:.2f} mm')
print(f'Final Seidels:        {lens.aberrations.seidels()}')

## 5. Full Performance Analysis

With the design converged we now characterise it with the complete set of diagnostics that
a lens designer would examine before handing a design to tolerancing:

- **Lens drawing** — layout with traced ray bundles
- **Spot diagram** — polychromatic spot positions at all three fields
- **RMS spot size vs field** — uniformity across the field
- **Ray fan** — tangential and sagittal aberration fans reveal the aberration content
- **Field curvature** — sagittal and tangential focal surfaces
- **Distortion** — percentage distortion across the field
- **MTF vs field** — the ultimate image-quality metric for a camera lens


In [ ]:
print('=== Final optimized Double Gauss — F/2.8 100 mm EFL ===')
lens.draw()


In [ ]:
print('=== Spot diagram (F, d, C lines — on-axis / 67% / full field) ===')
analysis.SpotDiagram(lens).view()


In [ ]:
print('=== RMS spot size vs normalised field height ===')
analysis.RmsSpotSizeVsField(lens).view()


In [ ]:
print('=== Ray fans — tangential and sagittal (F, d, C) ===')
analysis.RayFan(lens).view()


In [ ]:
print('=== Field curvature — sagittal (S) and tangential (T) surfaces ===')
analysis.FieldCurvature(lens).view()


In [ ]:
print('=== Distortion ===')
analysis.Distortion(lens).view()


In [ ]:
print('=== Polychromatic MTF vs field ===')
# Spatial frequencies in cycles/mm — 10, 20, 30, 40 lp/mm are typical camera lens targets
analysis.MTFvsField(lens, frequencies=[10, 20, 30, 40]).view()


## Conclusions

Starting from six flat glass plates we designed a polychromatic F/4 Double Gauss camera
lens through three successive optimisation phases:

1. **Phase 1 — Seidel**: Eight curvature variables (air-glass interfaces only) drove EFL
   to 100 mm and balanced the five primary aberrations simultaneously.  The optimizer found
   a plausible Double Gauss shape in under 100 iterations, with the cemented interfaces
   kept flat to prevent degenerate high-curvature solutions.  On-axis Seidel residual
   reached machine-epsilon (≈10⁻¹⁶) — a perfectly balanced paraxial design.

2. **Phase 2 — Polychromatic RMS spot**: Switching to a real-ray merit function and
   releasing the cemented interface curvatures and image distance drove the merit function
   from ~5 to ~0.12 — a **97% reduction** in 200 iterations.  The `wavelength='all'`
   option in `rms_spot_size` optimised all three wavelengths in a single operand.

3. **Phase 3 — Air-gap refinement**: Releasing four inter-element spacings gave
   additional degrees of freedom.  The small additional improvement (~5%) shows the
   design was already near a local minimum after Phase 2 — a common result when
   Seidel initialisation produces a well-shaped starting topology.

**Achieved performance (d-line RMS spot radius):**

| Field | After Phase 1 | After Phase 2 | After Phase 3 |
|-------|-------------|-------------|-------------|
| 0° (on-axis) | 0.002 mm | 0.101 mm | 0.097 mm |
| 10° (67 % field) | 0.287 mm | 0.176 mm | 0.169 mm |
| 15° (full field) | 2.229 mm | 0.117 mm | 0.113 mm |

The key insight from the Double Gauss is **symmetry as aberration cancellation**: by
mirroring the rear group onto the front group about the stop, all odd aberrations (coma,
distortion, lateral colour) are automatically suppressed.  The Seidel initialisation
exploits this symmetry to find a starting configuration that Phase 2 can refine quickly.

**Next steps:**

- To push toward F/2.8 or faster, replace Phase 1 with a global solver
  (`differential_evolution`) to escape the local minima that trap DLS at fast apertures.
- Use Tutorial 3c to tune the DLS controller (Gauss-Newton mode with Armijo line search)
  for faster Phase 2 convergence.
- Add `edge_thickness` operands to Phase 3 to enforce minimum glass thicknesses for
  manufacturability — a critical step before handing the design to tolerancing.
- Use Tutorial 3d to add `CheckpointObserver` to the long optimisation phases, enabling
  fault-tolerant overnight design sessions.
